# Project 1 - Medical Text Summarisation

## Data loading

In [4]:
## Import libraries
import os
import tarfile
from pathlib import Path
import pandas as pd

In [5]:
# Defining file paths

project_root = Path.cwd().parent
data_dir = project_root / "data"
archive_path = data_dir / "NLMCXR_reports.tgz"
extract_dir = data_dir / "nlmcxr_reports"

print("Project root:", project_root)
print("Data directory:", data_dir)
print("Archive path:", archive_path)
print("Archive exists:", archive_path.exists())
print("Extract folder:", extract_dir)

Project root: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser
Data directory: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data
Archive path: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\NLMCXR_reports.tgz
Archive exists: True
Extract folder: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports


In [6]:
# Extract the .tgz archive
extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print("Extraction complete.")
print("Files extracted to:", extract_dir)

Extraction complete.
Files extracted to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports


In [7]:
# Inspect the extracted files
all_files = list(extract_dir.rglob("*"))

print("Total extracted items:", len(all_files))

for file in all_files[:20]:
    print(file)

Total extracted items: 3956
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\10.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\100.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1000.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1001.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1002.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1003.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Pr

In [8]:
# Checking the file types that are present
xml_files = list(extract_dir.rglob("*.xml"))
txt_files = list(extract_dir.rglob("*.txt"))

print("Number of XML files:", len(xml_files))
print("Number of TXT files:", len(txt_files))

Number of XML files: 3955
Number of TXT files: 0


In [9]:
# Open to view a sample report
sample_file = xml_files[0]
print("Sample file path:", sample_file)

with open(sample_file, "r", encoding="utf-8") as f:
    content = f.read()

print(content[:2000])

Sample file path: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1.xml
<?xml version="1.0" encoding="utf-8"?>
<eCitation>
   <meta type="rr"/>
   <uId id="CXR1"/>
   <pmcId id="1"/>
   <docSource>CXR</docSource>
   <IUXRId id="1"/>
   <licenseType>open-access</licenseType>
   <licenseURL>http://creativecommons.org/licenses/by-nc-nd/4.0/</licenseURL>
   <ccLicense>byncnd</ccLicense>
   <articleURL/>
   <articleDate>2013-08-01</articleDate>
   <articleType>XR</articleType>
   <publisher>Indiana University</publisher>
   <title>Indiana University Chest X-ray Collection</title>
   <note>The data are drawn from multiple hospital systems.</note>
   <specialty>pulmonary diseases</specialty>
   <subset>CXR</subset>
   <MedlineCitation Owner="Indiana University" Status="supplied by publisher">
   
      <Article PubModel="Electronic">
      
         <Journal>
         
            <JournalIssue>
            
               <PubDate>

In [10]:
# Looking for whether the words FINDINGS and IMPRESSION appear in the sample file content

print("FINDINGS" in content)
print("IMPRESSION" in content)

True
True


### Summary Data Loading
I imported the required libraries, defined the dataset paths, extracted the compressed report archive, inspected the extracted files and opened a sample report to understand its structure. This helped confirmed whether the report files contain the 'FINDINGS' and 'IMPRESSION' fields needed for the summarisation task.

## Process Raw Reports into a flat file for modelling

In [11]:
# Import Libraries
import xml.etree.ElementTree as ET

# Define paths
output_file = data_dir / "processed_reports.csv"

print("Output fill will be:", output_file)

Output fill will be: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\processed_reports.csv


In [12]:
# Function to extract FINDINGS and IMPRESSION from the file
def extract_report_sections(xml_file):
    findings_text = None
    impression_text = None

    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        for abstract_text in root.iter("AbstractText"):
            label = abstract_text.attrib.get("Label", "").strip().upper()
            text = "".join(abstract_text.itertext()).strip()

            if label == "FINDINGS":
                findings_text = text
            elif label == "IMPRESSION":
                impression_text = text
            
        return findings_text, impression_text
    
    except Exception as e:
        return None, None


In [13]:
# Loooping through all report in the file to extract findings and impression

records = []

for xml_file in xml_files:
    findings, impression = extract_report_sections(xml_file)

    records.append({
        "file_name": xml_file.name,
        "findings": findings,
        "impression": impression
    })

print("Total records collected:", len(records))
print("First record:")
print(records[0])

Total records collected: 3955
First record:
{'file_name': '1.xml', 'findings': 'The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.', 'impression': 'Normal chest x-XXXX.'}


In [14]:
df = pd.DataFrame(records) # dataframe

print("DataFrame shape:", df.shape)
print(df.isna().sum())
df.head()

DataFrame shape: (3955, 3)
file_name     0
findings      0
impression    0
dtype: int64


,file_name,findings,impression
0,1.xml,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,10.xml,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,100.xml,Both lungs are clear and expanded. Heart and m...,No active disease.
3,1000.xml,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,1001.xml,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


In [15]:
# Cleaning the data
df = df.dropna(subset=["findings", "impression"])
df["findings"] = df["findings"].str.strip()
df["impression"] = df["impression"].str.strip()

df = df[(df["findings"] != "") & (df["impression"] != "")]

print("Shape after cleaning:", df.shape)
df.head()

Shape after cleaning: (3419, 3)


,file_name,findings,impression
0,1.xml,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,10.xml,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,100.xml,Both lungs are clear and expanded. Heart and m...,No active disease.
3,1000.xml,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,1001.xml,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


In [16]:
# Save the file
df.to_csv(output_file, index=False)

print("Flat file saved successfully.")
print("Saved to:", output_file)

Flat file saved successfully.
Saved to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\processed_reports.csv


In [17]:
# Inspect the results

print(df.info())
df.sample(5, random_state=42)

<class 'pandas.DataFrame'>
Index: 3419 entries, 0 to 3954
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   file_name   3419 non-null   str  
 1   findings    3419 non-null   str  
 2   impression  3419 non-null   str  
dtypes: str(3)
memory usage: 1.0 MB
None


,file_name,findings,impression
2277,3076.xml,"Frontal view kyphotic and rotated, low lung vo...","Low lung volumes, otherwise, no definite acute..."
3506,591.xml,The cardiomediastinal silhouette is within nor...,1. No acute cardiopulmonary process. 2. Age-in...
196,1177.xml,Normal heart. Clear lungs. Trachea midline. Sc...,No acute cardiopulmonary abnormality.
3821,879.xml,Cardiac and mediastinal contours are within no...,Negative chest x-XXXX.
1342,2222.xml,"Lungs are clear bilaterally, with no focal con...",1. Round opacity measuring 2 cm in diameter wi...


In [18]:
# Keeping only the findings and impression columns for modelling

model_df = df[["findings","impression"]].copy()
model_output_file = data_dir / "modeling_data.csv"

model_df.to_csv(model_output_file, index=False)

print("Modeling flat file saved to:", model_output_file)
model_df.head()

Modeling flat file saved to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\modeling_data.csv


,findings,impression
0,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,Both lungs are clear and expanded. Heart and m...,No active disease.
3,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


### Summary: Processing raw data

I processed the raw XML reports into a flat file for modeling. I searched through all the report files, parsed each XML document, extracted the FINDINGS and IMPRESSION sections, stored them in a pandas DataFrame, removed incomplete records and saved the cleaned dataset as a CSV file

## Train-Test Split

In [19]:
# import function
from sklearn.model_selection import train_test_split

# Define input and output variables
X = model_df["findings"]
y = model_df["impression"]


# Split data into train and test 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# check the size of splits datasets
print("Training set size:", len(X_train))
print("Test set size:", len(X_test))


Training set size: 2735
Test set size: 684


## Feature Creation

In [20]:
# Import the libraries
from transformers import T5Tokenizer

# load the tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Create a preprocessing funtion
def preprocess_function(examples):
    # Include summarization prefix for each finding text
    inputs = ["summarize: " +  str(text).strip() for text in examples["findings"]]

    # Clean target summaries
    targets = [str(text).strip() for text in examples["impression"]]

    # Tokenize the input findings text
    model_inputs = tokenizer( 
        inputs,
        max_length=512,
        truncation=True,
    )

    # Tokenize the target impression text 
    labels = tokenizer( 
        text_target=targets,
        max_length=128,
        truncation=True,
    )
    
    # Add tokenized target as labels
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

    

In [21]:
# Viewing the output of feature creation
sample_examples = {
    "findings": [X_train.iloc[0]],
    "impression": [y_train.iloc[0]]
}

sample_features = preprocess_function(sample_examples)

print("Keys in train encodings:", sample_features.keys())
print("First input_ids sample:", sample_features["input_ids"][0][:20])
print("First attention_mask sample:", sample_features["attention_mask"][0][:20])
print("First label input_ids sample:", sample_features["input_ids"][0][:20])

Keys in train encodings: KeysView({'input_ids': [[21603, 10, 6219, 812, 441, 1389, 6790, 5, 749, 1109, 295, 1413, 2248, 10646, 11, 150, 26, 4885, 3, 32, 5379, 2197, 33, 1936, 437, 4993, 3631, 5, 290, 19, 3, 9, 209, 2446, 150, 26, 4885, 3, 32, 5379, 485, 16, 8, 269, 583, 10775, 60, 2532, 3, 4, 4, 4, 4, 6, 1936, 437, 4993, 6498, 5, 71, 26530, 447, 90, 1938, 16, 8, 269, 4548, 3, 11846, 15, 3475, 1126, 12, 1884, 6498, 5, 465, 3, 4788, 9709, 13577, 11733, 42, 18919, 8888, 21783, 226, 5, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[14490, 7, 13, 26530, 447, 3, 20602, 7, 159, 28, 1936, 1413, 2248, 10646, 11, 150, 26, 4885, 3, 32, 5379, 2197, 6, 3, 4, 4, 4, 4, 9085, 12498, 1215, 9, 2110, 115, 257, 30, 6658, 1112, 13, 26530, 447, 3, 20

### Feature Creation Summary
I created a reusable preprocessiong function to tokenize the findings and impression for the transformer model. This function convert raw text into tokenized input IDS, attention masks and labels. I used this function so that the same logic be reused during model training making it easier to make improvements later and manage

## Model Building - Fine-Tuning T5 Small

In [22]:
# Import libraries for model building
from datasets import Dataset
from transformers import (
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# Convert train and test data into DataFrames
train_df = pd.DataFrame({
    "findings": X_train,
    "impression": y_train
}).reset_index(drop=True)

test_df = pd.DataFrame({
    "findings": X_test,
    "impression": y_test
}).reset_index(drop=True)

# View train and test dataframe
print("Traing DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)
train_df.head()

Traing DataFrame shape: (2735, 2)
Test DataFrame shape: (684, 2)


,findings,impression
0,Heart size within normal limits. Prominent int...,Findings of cystic fibrosis with increased int...
1,The lungs are clear. The cardiomediastinal sil...,Negative chest .
2,"Lung volumes are low. The heart is large, and ...",Congestive heart failure with bibasilar pulmon...
3,"Consolidation, atelectasis, and costophrenic X...",Cleared left lower lobe airspace disease with ...
4,"Heart size, mediastinal contour, and pulmonary...",No acute cardiopulmonary abnormality.


In [23]:
# Conver pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['findings', 'impression'],
    num_rows: 2735
})
Dataset({
    features: ['findings', 'impression'],
    num_rows: 684
})


In [24]:
# Apply the feature creation preprocessing function
tokenized_train = train_dataset.map(
    preprocess_function, 
    batched=True,
    remove_columns=train_dataset.column_names    
)
tokenized_test = test_dataset.map(
    preprocess_function, 
    batched=True,
    remove_columns=test_dataset.column_names
)

# View tokenized output
print("Column names:", tokenized_train.column_names)
print("Key in first example:", tokenized_train[0].keys())
print("First example:", tokenized_train[0])

print("Sample inputs_ids:", tokenized_train[0]["input_ids"][:10])
if "labels" in tokenized_train[0]:
    print("Sample labels:", tokenized_train[0]["labels"][:10])
else:
    print("No labels key found in tokenized_train[0]")


Map:   0%|          | 0/2735 [00:00<?, ? examples/s]

Map:   0%|          | 0/684 [00:00<?, ? examples/s]

Column names: ['input_ids', 'attention_mask', 'labels']
Key in first example: dict_keys(['input_ids', 'attention_mask', 'labels'])
First example: {'input_ids': [21603, 10, 6219, 812, 441, 1389, 6790, 5, 749, 1109, 295, 1413, 2248, 10646, 11, 150, 26, 4885, 3, 32, 5379, 2197, 33, 1936, 437, 4993, 3631, 5, 290, 19, 3, 9, 209, 2446, 150, 26, 4885, 3, 32, 5379, 485, 16, 8, 269, 583, 10775, 60, 2532, 3, 4, 4, 4, 4, 6, 1936, 437, 4993, 6498, 5, 71, 26530, 447, 90, 1938, 16, 8, 269, 4548, 3, 11846, 15, 3475, 1126, 12, 1884, 6498, 5, 465, 3, 4788, 9709, 13577, 11733, 42, 18919, 8888, 21783, 226, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [14490, 7, 13, 26530, 447, 3, 20602, 7, 159, 28, 1936, 1413, 2248, 10646, 11, 150, 26, 4885, 3, 32

In [25]:

# Load the T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Data collator for sequence-to-sequence tasks
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/t5-small-radiology",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="../logs",
    logging_steps=50,
    report_to="none",
    predict_with_generate=True
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

# Train
trainer.train()

# Save the fine-tuned model and tokenizer
save_path = "../models/t5-small-radiology-final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved to:", save_path)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\med_venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.269635,1.976228


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: ../models/t5-small-radiology-final


### Model fine tuning summary
The 't5-small' model fine-tuned successfuly on the dataset for 1 epoch. The final training loss was 2.2696 and the validation loss was 1.9762. The fine-tune model and tokenizer saved for later evaluation and deployment

## Model Performance evaluation

In [27]:
# Import libraries
from rouge_score import rouge_scorer
import numpy as np

# Prediction function
def generate_predictions(model, tokenizer, texts):
    predictions = []
    
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        ouput_ids = model.generate(
            **inputs,
            max_length=128
        )

        pred = tokenizer.decode(ouput_ids[0], skip_special_tokens=True)
        predictions.append(pred)

    return predictions

test_texts = test_df["findings"].tolist()
true_summaries = test_df["impression"].tolist()

pred_summaries = generate_predictions(model, tokenizer, test_texts)



# Comput Rouge-1 score
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

scores = []

for pred, true in zip(pred_summaries, true_summaries):
    score = scorer.score(true, pred)
    scores.append(score['rouge1'].fmeasure)

avg_rouge = np.mean(scores)

print("Average ROUGE-1 G1 score:", avg_rouge)



Average ROUGE-1 G1 score: 0.24720890745578344


In [28]:
# Show 3 predictions
for i in range(3):
    print("\n==================================")
    print("FINDINGS:")
    print(test_texts[i])

    print("\nPREDICTED IMPRESSION:")
    print(pred_summaries[i])

    print("\nACTUAL IMPRESSION")
    print(true_summaries[i])


FINDINGS:
Frontal view kyphotic and rotated, low lung volumes with bronchovascular crowding. Otherwise, no definite airspace consolidation or pleural effusion. Accounting for technical factors heart size XXXX borderline enlarged.

PREDICTED IMPRESSION:
bronchovascular crowding. Otherwise, no definitive airspace consolidation or pleural effusion. Accounting for technical factors heart size XXXX borderline enlarged.

ACTUAL IMPRESSION
Low lung volumes, otherwise, no definite acute findings.

FINDINGS:
The cardiomediastinal silhouette is within normal limits. The lungs are clear without areas of focal consolidation. There is a calcified granuloma within the left lung base. There is suggestion of a deep sulcus sign on the right. No definite pleural line of pneumothorax visualized. There is age-indeterminate wedging of several midthoracic vertebral bodies.

PREDICTED IMPRESSION:
. No acute cardiopulmonary system. No acute cardiopulmonary abnormality visualized.

ACTUAL IMPRESSION
1. No acu

### Summary Model Performance Evaluation Summary
The model was evaluated on the test dataset using ROUGE-1 F1 score obtaining an Average ROUGE-1 G1 score of 0.2472. This is below the require 0.3 or more for this project so will need to retrain. Additionally, sample predictions were inspected to for model performance and was less concise. This to be address in the retrain